In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h3
from dotenv import load_dotenv

from backend.database.db_io import connect_db, get_city_center

load_dotenv()

# ─── Parameters ─────────────────────────────────────────────────────────────────────────────
CITY_ID       = 1        # Default: Madrid. Swap to any city_id.
H3_RESOLUTION = 7        # 7 ≈ 1.2 km cells → ~20-40 bubbles; 8 ≈ 0.5 km → denser
MIN_VOLUME    = 50       # Drop hexes with fewer total trips than this
BUBBLE_SCALE  = 1.0      # Radius multiplier — tune visually after first run
BASE_RADIUS   = 18       # Base scatter size before volume scaling
OUTPUT_DIR    = Path('frontend/public/landing')
FIG_W, FIG_H  = 6, 5
DPI           = 150
COLOR         = '#027A76'
BG_COLOR      = '#FBF6EF'
BBOX_COLOR    = '#003849'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR.resolve()}')

In [ ]:
conn = connect_db()

# City center for bbox + coordinate reference
center_data = get_city_center(conn, CITY_ID)
if center_data is None:
    raise ValueError(f'No center coordinates for city_id={CITY_ID}')
center_lat, center_lon, _ = center_data
print(f'City center: ({center_lat:.4f}, {center_lon:.4f})')

# Origin counts per node
with conn.cursor() as cur:
    cur.execute("""
        SELECT n.lat, n.lon, COUNT(*) AS cnt
        FROM trips t
        JOIN nodes n ON n.id = t.origin_node
        WHERE t.city_id = %s
          AND t.origin_node IS NOT NULL
          AND n.lat IS NOT NULL AND n.lon IS NOT NULL
        GROUP BY n.lat, n.lon
    """, (CITY_ID,))
    origins = cur.fetchall()

# Destination counts per node
with conn.cursor() as cur:
    cur.execute("""
        SELECT n.lat, n.lon, COUNT(*) AS cnt
        FROM trips t
        JOIN nodes n ON n.id = t.dest_node
        WHERE t.city_id = %s
          AND t.dest_node IS NOT NULL
          AND n.lat IS NOT NULL AND n.lon IS NOT NULL
        GROUP BY n.lat, n.lon
    """, (CITY_ID,))
    destinations = cur.fetchall()

conn.close()
print(f'Raw origin nodes: {len(origins)}, destination nodes: {len(destinations)}')

# Aggregate into H3 hexes (origins + destinations combined)
hex_volumes: dict = defaultdict(int)
for lat, lon, cnt in list(origins) + list(destinations):
    cell = h3.latlng_to_cell(float(lat), float(lon), H3_RESOLUTION)
    hex_volumes[cell] += int(cnt)

# Filter by MIN_VOLUME, build DataFrame with hex centroid coordinates
rows = []
for cell, vol in hex_volumes.items():
    if vol >= MIN_VOLUME:
        clat, clon = h3.cell_to_latlng(cell)
        rows.append({'lat': clat, 'lon': clon, 'volume': vol})

bubbles_df = pd.DataFrame(rows)
print(f'Hexes after MIN_VOLUME={MIN_VOLUME} filter: {len(bubbles_df)}')
if not bubbles_df.empty:
    print(bubbles_df['volume'].describe().to_string())